In [1]:
# CELL 01 — Imports + paths
# Mục tiêu: chuẩn bị thư viện, đường dẫn và kiểm tra 8 file raw.

import os
import pandas as pd
import pyarrow.parquet as pq
import numpy as np

RAW_DIR = "../data/raw"
INTERIM_DIR = "../data/interim"
os.makedirs(INTERIM_DIR, exist_ok=True)

files_expected = [
    "train.csv", "test.csv", "stores.csv", "items.csv",
    "oil.csv", "holidays_events.csv", "transactions.csv",
    "sample_submission.csv",
]

print("RAW_DIR:", os.path.abspath(RAW_DIR))
print("INTERIM_DIR:", os.path.abspath(INTERIM_DIR))

missing = [f for f in files_expected if not os.path.exists(os.path.join(RAW_DIR, f))]
if missing:
    print("\n[FAIL] Thiếu file:")
    for f in missing:
        print(" -", f)
else:
    print("\n[PASS] Đã tìm thấy toàn bộ 8 file dữ liệu chính.")


RAW_DIR: /home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast/data/raw
INTERIM_DIR: /home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast/data/interim

[PASS] Đã tìm thấy toàn bộ 8 file dữ liệu chính.


In [2]:
# CELL 02 — Data Inventory
# Mục tiêu: biết kích thước từng CSV mà không load train/test vào RAM.

inventory = []
for filename in files_expected:
    path = os.path.join(RAW_DIR, filename)
    if not os.path.exists(path):
        continue
    size_mb = os.path.getsize(path) / (1024 ** 2)
    with open(path, "rb") as f:
        n_lines = sum(1 for _ in f) - 1
    inventory.append({
        "file": filename,
        "size_mb": round(size_mb, 2),
        "rows_estimated": n_lines,
    })

inventory_df = pd.DataFrame(inventory).sort_values(
    "size_mb", ascending=False
).reset_index(drop=True)

display(inventory_df)
print("\nKết quả cần quan sát:")
print("- train.csv là bảng lớn nhất và cần đọc theo chunks.")
print("- stores/items/oil/holidays/transactions nhỏ hơn đáng kể.")
print("- test.csv cũng nên dùng usecols khi đọc.")


,file,size_mb,rows_estimated
0,train.csv,4765.94,125497040
1,test.csv,120.32,3370464
2,sample_submission.csv,38.57,3370464
3,transactions.csv,1.48,83488
4,items.csv,0.10,4100
5,holidays_events.csv,0.02,350
6,oil.csv,0.02,1218
7,stores.csv,0.00,54



Kết quả cần quan sát:
- train.csv là bảng lớn nhất và cần đọc theo chunks.
- stores/items/oil/holidays/transactions nhỏ hơn đáng kể.
- test.csv cũng nên dùng usecols khi đọc.


In [3]:
# CELL 03 — Load các bảng tham chiếu nhỏ
# Mục tiêu: load các bảng nhỏ và kiểm tra schema ban đầu.

items = pd.read_csv(os.path.join(RAW_DIR, "items.csv"))
stores = pd.read_csv(os.path.join(RAW_DIR, "stores.csv"))
oil = pd.read_csv(os.path.join(RAW_DIR, "oil.csv"))
holidays = pd.read_csv(os.path.join(RAW_DIR, "holidays_events.csv"))
transactions = pd.read_csv(os.path.join(RAW_DIR, "transactions.csv"))

print("=== SHAPES ===")
for name, df in {
    "items": items,
    "stores": stores,
    "oil": oil,
    "holidays": holidays,
    "transactions": transactions,
}.items():
    print(f"{name:12s}: {df.shape}")

print("\n=== COLUMNS ===")
for name, df in {
    "items": items,
    "stores": stores,
    "oil": oil,
    "holidays": holidays,
    "transactions": transactions,
}.items():
    print(f"\n{name}:")
    print(df.columns.tolist())

print("\n=== SAMPLE ===")
display(items.head())
display(stores.head())
display(oil.head())
display(holidays.head())
display(transactions.head())


=== SHAPES ===
items       : (4100, 4)
stores      : (54, 5)
oil         : (1218, 2)
holidays    : (350, 6)
transactions: (83488, 3)

=== COLUMNS ===

items:
['item_nbr', 'family', 'class', 'perishable']

stores:
['store_nbr', 'city', 'state', 'type', 'cluster']

oil:
['date', 'dcoilwtico']

holidays:
['date', 'type', 'locale', 'locale_name', 'description', 'transferred']

transactions:
['date', 'store_nbr', 'transactions']

=== SAMPLE ===


,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [4]:
# CELL 04 — Reference-table integrity
# Mục tiêu: kiểm tra key trước khi JOIN để tránh nhân bản dòng.

items_dup = items.duplicated(["item_nbr"]).sum()
stores_dup = stores.duplicated(["store_nbr"]).sum()

oil = oil.copy()
oil["date"] = pd.to_datetime(oil["date"], errors="coerce")
oil_dup = oil.duplicated(["date"]).sum()

transactions = transactions.copy()
transactions["date"] = pd.to_datetime(transactions["date"], errors="coerce")
trans_dup = transactions.duplicated(["date", "store_nbr"]).sum()

holidays = holidays.copy()
holidays["date"] = pd.to_datetime(holidays["date"], errors="coerce")
hol_dup_date = holidays.duplicated(["date"]).sum()
hol_dup_full = holidays.duplicated(
    ["date", "locale", "locale_name", "description"]
).sum()

print("=== DUPLICATE KEY CHECK ===")
print(f"items:        duplicate (item_nbr) = {items_dup:,}")
print(f"stores:       duplicate (store_nbr) = {stores_dup:,}")
print(f"oil:          duplicate (date) = {oil_dup:,}")
print(f"transactions: duplicate (date, store_nbr) = {trans_dup:,}")
print(f"holidays:     duplicate (date) = {hol_dup_date:,}")
print(f"holidays:     duplicate full logical key = {hol_dup_full:,}")

print("\n=== INTERPRETATION ===")
print("- items/store/oil/transactions nên có key tương ứng unique.")
print("- holidays có thể có nhiều event cùng date; đó không tự động là lỗi.")

if items_dup == 0 and stores_dup == 0 and oil_dup == 0 and trans_dup == 0:
    print("\n[PASS] Các key chính của bảng tham chiếu không bị duplicate.")
else:
    print("\n[FAIL] Có key bị duplicate; cần xử lý trước khi JOIN.")


=== DUPLICATE KEY CHECK ===
items:        duplicate (item_nbr) = 0
stores:       duplicate (store_nbr) = 0
oil:          duplicate (date) = 0
transactions: duplicate (date, store_nbr) = 0
holidays:     duplicate (date) = 38
holidays:     duplicate full logical key = 0

=== INTERPRETATION ===
- items/store/oil/transactions nên có key tương ứng unique.
- holidays có thể có nhiều event cùng date; đó không tự động là lỗi.

[PASS] Các key chính của bảng tham chiếu không bị duplicate.


In [5]:
# CELL 05 — Train profile theo chunks
# Mục tiêu:
# 1) kiểm tra thứ tự date toàn file, kể cả giữa các chunk
# 2) duplicate theo (date, store_nbr, item_nbr)
# 3) profile theo ngày
# 4) negative sales và onpromotion missing/true

chunk_size = 1_000_000
train_cols = ["date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"]
train_key = ["date", "store_nbr", "item_nbr"]

def profile_days(block):
    b = block.assign(
        is_neg=block["unit_sales"] < 0,
        promo_nan=block["onpromotion"].isna(),
        promo_true=block["onpromotion"] == True,
    )
    return b.groupby("date").agg(
        n_rows=("item_nbr", "size"),
        n_stores=("store_nbr", "nunique"),
        n_items=("item_nbr", "nunique"),
        sales_sum=("unit_sales", "sum"),
        n_negative=("is_neg", "sum"),
        n_promo_nan=("promo_nan", "sum"),
        n_promo_true=("promo_true", "sum"),
    )

dtype_map = {
    "store_nbr": "int16",
    "item_nbr": "int32",
    "unit_sales": "float32",
    "onpromotion": "boolean",
}

profiles = []
dup_total = 0
sorted_ok = True
prev_last_date = None
train_items = set()
tail = None

reader = (batch.to_pandas() for batch in pq.ParquetFile("../data/processed/train.parquet").iter_batches(batch_size=chunk_size))

for chunk_id, chunk in enumerate(reader):
    chunk["date"] = chunk["date"].astype(str)

    first_date = chunk["date"].iloc[0]
    last_date = chunk["date"].iloc[-1]

    if not chunk["date"].is_monotonic_increasing:
        sorted_ok = False
    if prev_last_date is not None and first_date < prev_last_date:
        sorted_ok = False
    prev_last_date = last_date

    if tail is not None:
        chunk = pd.concat([tail, chunk], ignore_index=True)

    last_date = chunk["date"].iloc[-1]
    is_last = chunk["date"] == last_date
    tail = chunk[is_last].copy()
    body = chunk[~is_last].copy()

    if len(body):
        dup_total += body.duplicated(train_key).sum()
        train_items.update(body["item_nbr"].unique())
        profiles.append(profile_days(body))

    print(
        f"Processed chunk {chunk_id + 1:,} | "
        f"rows={len(chunk):,} | last_date={last_date}"
    )

if tail is not None and len(tail):
    dup_total += tail.duplicated(train_key).sum()
    train_items.update(tail["item_nbr"].unique())
    profiles.append(profile_days(tail))

daily_profile = pd.concat(profiles).sort_index()
daily_profile.index.name = "date"
profile_path = os.path.join(INTERIM_DIR, "train_daily_profile.csv")
daily_profile.to_csv(profile_path)

print("\n=== TRAIN PROFILE RESULT ===")
print(f"File sắp xếp theo date           : {sorted_ok}")
print(f"Mỗi ngày xuất hiện đúng 1 lần   : {daily_profile.index.is_unique}")
print(f"Số ngày                          : {len(daily_profile):,}")
print(f"Tổng số dòng                    : {daily_profile['n_rows'].sum():,}")
print(f"Số dòng trùng (date,store,item) : {dup_total:,}")
print(f"Số item khác nhau trong train   : {len(train_items):,}")
print(f"Đã lưu                           : {profile_path}")

display(daily_profile.head())

print("\n=== INTERPRETATION ===")
print("- sorted_ok phải là True cho toàn file, gồm cả ranh giới giữa chunks.")
print("- duplicate key phải là 0.")
print("- Không được diễn giải 'không có row' là 'sales = 0'.")


Processed chunk 1 | rows=1,000,000 | last_date=2013-01-26
Processed chunk 2 | rows=1,032,562 | last_date=2013-02-19
Processed chunk 3 | rows=1,039,414 | last_date=2013-03-15
Processed chunk 4 | rows=1,035,981 | last_date=2013-04-08
Processed chunk 5 | rows=1,000,108 | last_date=2013-05-01
Processed chunk 6 | rows=1,021,348 | last_date=2013-05-24
Processed chunk 7 | rows=1,013,230 | last_date=2013-06-15
Processed chunk 8 | rows=1,028,689 | last_date=2013-07-07
Processed chunk 9 | rows=1,031,353 | last_date=2013-07-30
Processed chunk 10 | rows=1,010,656 | last_date=2013-08-21
Processed chunk 11 | rows=1,000,841 | last_date=2013-09-11
Processed chunk 12 | rows=1,033,444 | last_date=2013-10-03
Processed chunk 13 | rows=1,025,969 | last_date=2013-10-25
Processed chunk 14 | rows=1,012,575 | last_date=2013-11-15
Processed chunk 15 | rows=1,013,291 | last_date=2013-12-05
Processed chunk 16 | rows=1,018,537 | last_date=2013-12-24
Processed chunk 17 | rows=1,039,512 | last_date=2014-01-12
Proces

,n_rows,n_stores,n_items,sales_sum,n_negative,n_promo_nan,n_promo_true
date,,,,,,,
2013-01-01,578,1,578,2511.618896,0,578,0
2013-01-02,41676,46,1583,496092.406250,1,41676,0
2013-01-03,40100,46,1586,361429.218750,5,40100,0
2013-01-04,40048,46,1586,354459.687500,4,40048,0
2013-01-05,42534,46,1595,477350.125000,1,42534,0



=== INTERPRETATION ===
- sorted_ok phải là True cho toàn file, gồm cả ranh giới giữa chunks.
- duplicate key phải là 0.
- Không được diễn giải 'không có row' là 'sales = 0'.


In [6]:
# CELL 06 — Daily profile + calendar coverage
# Mục tiêu:
# - Tạo dataframe `daily` từ daily_profile
# - Kiểm tra ngày đầu/cuối
# - Tìm các ngày bị thiếu trong khoảng train
# - Không load lại toàn bộ train.csv

daily = daily_profile.copy()

# Đảm bảo index là DatetimeIndex
daily.index = pd.to_datetime(daily.index)
daily.index.name = "date"

print("=== DAILY PROFILE ===")
print(f"Ngày bắt đầu train : {daily.index.min().date()}")
print(f"Ngày kết thúc train : {daily.index.max().date()}")
print(f"Số ngày quan sát   : {len(daily):,}")

# Tạo toàn bộ lịch liên tục từ ngày đầu đến ngày cuối
expected_dates = pd.date_range(
    daily.index.min(),
    daily.index.max(),
    freq="D"
)

train_missing_dates = expected_dates.difference(
    daily.index
)

print(f"\nSố ngày theo calendar kỳ vọng : {len(expected_dates):,}")
print(f"Số ngày thực sự có trong train: {daily.index.nunique():,}")
print(f"Số ngày bị thiếu               : {len(train_missing_dates):,}")

if len(train_missing_dates) > 0:
    print("\nCác ngày bị thiếu:")
    display(pd.DataFrame({"missing_date": train_missing_dates}))
else:
    print("\n[PASS] Không có ngày bị thiếu trong khoảng train.")

# Lưu lại daily để các cell sau có thể dùng
daily_path = os.path.join(
    INTERIM_DIR,
    "train_daily.csv"
)

daily.to_csv(daily_path)

print(f"\nĐã lưu: {daily_path}")

print("\n=== SAMPLE ===")
display(daily.head())
display(daily.tail())


=== DAILY PROFILE ===
Ngày bắt đầu train : 2013-01-01
Ngày kết thúc train : 2017-08-15
Số ngày quan sát   : 1,684

Số ngày theo calendar kỳ vọng : 1,688
Số ngày thực sự có trong train: 1,684
Số ngày bị thiếu               : 4

Các ngày bị thiếu:


,missing_date
0,2013-12-25
1,2014-12-25
2,2015-12-25
3,2016-12-25



Đã lưu: ../data/interim/train_daily.csv

=== SAMPLE ===


,n_rows,n_stores,n_items,sales_sum,n_negative,n_promo_nan,n_promo_true
date,,,,,,,
2013-01-01,578,1,578,2511.618896,0,578,0
2013-01-02,41676,46,1583,496092.406250,1,41676,0
2013-01-03,40100,46,1586,361429.218750,5,40100,0
2013-01-04,40048,46,1586,354459.687500,4,40048,0
2013-01-05,42534,46,1595,477350.125000,1,42534,0


,n_rows,n_stores,n_items,sales_sum,n_negative,n_promo_nan,n_promo_true
date,,,,,,,
2017-08-11,104599,54,3780,826373.7500,3,0,14179
2017-08-12,105192,54,3761,792630.5625,6,0,8312
2017-08-13,106104,54,3773,865639.6875,5,0,9283
2017-08-14,102750,54,3762,760922.3750,9,0,8043
2017-08-15,102812,54,3771,762661.9375,6,0,10605


In [7]:
# CELL 07 — Sales & Promotion integrity
# Mục tiêu:
# - Tổng hợp negative sales
# - Kiểm tra explicit zero trong train
# - Tổng số promotion True
# - Tỷ lệ missing promotion
# - Kiểm tra phân bố sales theo ngày
#
# Lưu ý:
# "unit_sales == 0" ở đây chỉ là các ROW thực sự tồn tại trong train.
# Không có row KHÔNG đồng nghĩa với sales = 0.

total_rows = 0
negative_sales = 0
zero_sales = 0
promo_true = 0
promo_missing = 0

sales_min = np.inf
sales_max = -np.inf

daily_sales_check = []

reader = (batch.to_pandas() for batch in pq.ParquetFile("../data/processed/train.parquet").iter_batches(batch_size=chunk_size))

for chunk in reader:

    total_rows += len(chunk)

    negative_sales += int(
        (chunk["unit_sales"] < 0).sum()
    )

    zero_sales += int(
        (chunk["unit_sales"] == 0).sum()
    )

    promo_true += int(
        (chunk["onpromotion"] == True).sum()
    )

    promo_missing += int(
        chunk["onpromotion"].isna().sum()
    )

    if len(chunk):
        sales_min = min(
            sales_min,
            chunk["unit_sales"].min()
        )

        sales_max = max(
            sales_max,
            chunk["unit_sales"].max()
        )

print("=== SALES CHECK ===")
print(f"Tổng số dòng              : {total_rows:,}")
print(f"unit_sales nhỏ nhất       : {sales_min:,.4f}")
print(f"unit_sales lớn nhất       : {sales_max:,.4f}")
print(f"Dòng unit_sales < 0       : {negative_sales:,}")
print(f"Dòng unit_sales == 0      : {zero_sales:,}")

print("\n=== PROMOTION CHECK ===")
print(f"onpromotion == True       : {promo_true:,}")
print(f"onpromotion bị thiếu      : {promo_missing:,}")

print("\n=== TỶ LỆ ===")
print(
    f"Negative sales rate       : "
    f"{negative_sales / total_rows:.4%}"
)

print(
    f"Explicit zero sales rate  : "
    f"{zero_sales / total_rows:.4%}"
)

print(
    f"Promotion True rate       : "
    f"{promo_true / total_rows:.4%}"
)

print(
    f"Promotion missing rate    : "
    f"{promo_missing / total_rows:.4%}"
)

print("\n=== INTERPRETATION ===")
print("- Negative sales cần được phân tích trước khi model hóa.")
print("- Explicit zero chỉ nói về những row thực sự có trong train.")
print("- Missing promotion không nên tự động đổi thành False ở bước checking.")
print("- Không được dùng kết quả này để kết luận rằng toàn bộ grid có/không có sales = 0.")


=== SALES CHECK ===
Tổng số dòng              : 125,497,040
unit_sales nhỏ nhất       : -15,372.0000
unit_sales lớn nhất       : 89,440.0000
Dòng unit_sales < 0       : 7,795
Dòng unit_sales == 0      : 0

=== PROMOTION CHECK ===
onpromotion == True       : 7,810,622
onpromotion bị thiếu      : 21,657,651

=== TỶ LỆ ===
Negative sales rate       : 0.0062%
Explicit zero sales rate  : 0.0000%
Promotion True rate       : 6.2237%
Promotion missing rate    : 17.2575%

=== INTERPRETATION ===
- Negative sales cần được phân tích trước khi model hóa.
- Explicit zero chỉ nói về những row thực sự có trong train.
- Missing promotion không nên tự động đổi thành False ở bước checking.
- Không được dùng kết quả này để kết luận rằng toàn bộ grid có/không có sales = 0.


In [8]:
# CELL 08 — Test integrity (strict)
# Mục tiêu:
# - Kiểm tra tổng số dòng
# - Kiểm tra missing/null ở các cột quan trọng
# - Kiểm tra duplicate ID trong từng chunk
# - Kiểm tra duplicate ID giữa các chunk
# - Kiểm tra duplicate composite key trong từng chunk
# - Kiểm tra duplicate composite key giữa các chunk
# - Kiểm tra date range
# - Kiểm tra số store/item/date
#
# Lưu ý:
# Test có ~3.37M dòng nên vẫn đọc theo chunks.
# Dùng seen_ids / seen_keys để bắt duplicate giữa các chunk.
# Đồng thời dùng duplicated() trong từng chunk để không bỏ sót
# duplicate xảy ra ngay bên trong một chunk.

test_rows = 0

# Duplicate
test_id_duplicate = 0
test_key_duplicate = 0

# Missing
missing_id = 0
missing_date = 0
missing_store = 0
missing_item = 0

# Coverage
test_stores = set()
test_items = set()
test_dates = set()

# Dùng để phát hiện duplicate giữa các chunk
seen_ids = set()
seen_keys = set()

test_min_date = None
test_max_date = None


test_reader = pd.read_csv(
    os.path.join(RAW_DIR, "test.csv"),
    usecols=[
        "id",
        "date",
        "store_nbr",
        "item_nbr",
        "onpromotion"
    ],
    dtype={
        "id": "int64",
        "store_nbr": "int16",
        "item_nbr": "int32"
    },
    chunksize=500_000,
    low_memory=False
)


for chunk_id, chunk in enumerate(test_reader, start=1):

    test_rows += len(chunk)

    # =========================================================
    # 1. MISSING / NULL CHECK
    # =========================================================

    missing_id += int(chunk["id"].isna().sum())
    missing_date += int(chunk["date"].isna().sum())
    missing_store += int(chunk["store_nbr"].isna().sum())
    missing_item += int(chunk["item_nbr"].isna().sum())


    # =========================================================
    # 2. DUPLICATE ID TRONG CHÍNH CHUNK
    # =========================================================

    duplicate_id_inside = chunk["id"].duplicated().sum()

    test_id_duplicate += int(duplicate_id_inside)


    # =========================================================
    # 3. DUPLICATE ID GIỮA CÁC CHUNK
    # =========================================================

    duplicate_id_across = chunk["id"].isin(seen_ids).sum()

    test_id_duplicate += int(duplicate_id_across)

    seen_ids.update(
        chunk["id"].dropna().tolist()
    )


    # =========================================================
    # 4. DUPLICATE COMPOSITE KEY TRONG CHÍNH CHUNK
    # =========================================================

    composite_cols = [
        "date",
        "store_nbr",
        "item_nbr"
    ]

    duplicate_key_inside = chunk.duplicated(
        subset=composite_cols
    ).sum()

    test_key_duplicate += int(duplicate_key_inside)


    # =========================================================
    # 5. DUPLICATE COMPOSITE KEY GIỮA CÁC CHUNK
    # =========================================================

    # Tạo tuple key
    keys = list(
        zip(
            chunk["date"],
            chunk["store_nbr"],
            chunk["item_nbr"]
        )
    )

    duplicate_key_across = sum(
        key in seen_keys
        for key in keys
    )

    test_key_duplicate += int(
        duplicate_key_across
    )

    seen_keys.update(keys)


    # =========================================================
    # 6. COVERAGE
    # =========================================================

    test_stores.update(
        chunk["store_nbr"].dropna().unique()
    )

    test_items.update(
        chunk["item_nbr"].dropna().unique()
    )

    test_dates.update(
        chunk["date"].dropna().unique()
    )


    # =========================================================
    # 7. DATE RANGE
    # =========================================================

    if len(chunk):

        chunk_min_date = chunk["date"].min()
        chunk_max_date = chunk["date"].max()

        if (
            test_min_date is None
            or chunk_min_date < test_min_date
        ):
            test_min_date = chunk_min_date

        if (
            test_max_date is None
            or chunk_max_date > test_max_date
        ):
            test_max_date = chunk_max_date


    print(
        f"Chunk {chunk_id:>2} checked | "
        f"rows = {len(chunk):,}"
    )


# =============================================================
# 8. RESULT
# =============================================================

print("\n" + "=" * 55)
print("TEST INTEGRITY RESULT")
print("=" * 55)

print(f"Tổng số dòng                    : {test_rows:,}")

print("\n--- DUPLICATE ---")

print(
    f"Duplicate id                    : "
    f"{test_id_duplicate:,}"
)

print(
    f"Duplicate (date, store, item)   : "
    f"{test_key_duplicate:,}"
)

print("\n--- MISSING KEY VALUES ---")

print(
    f"Missing id                      : "
    f"{missing_id:,}"
)

print(
    f"Missing date                    : "
    f"{missing_date:,}"
)

print(
    f"Missing store_nbr               : "
    f"{missing_store:,}"
)

print(
    f"Missing item_nbr                : "
    f"{missing_item:,}"
)

print("\n--- COVERAGE ---")

print(
    f"Số store khác nhau              : "
    f"{len(test_stores):,}"
)

print(
    f"Số item khác nhau               : "
    f"{len(test_items):,}"
)

print(
    f"Số ngày khác nhau               : "
    f"{len(test_dates):,}"
)

print("\n--- DATE RANGE ---")

print(
    f"Ngày bắt đầu test               : "
    f"{test_min_date}"
)

print(
    f"Ngày kết thúc test              : "
    f"{test_max_date}"
)


# =============================================================
# 9. PASS / FAIL
# =============================================================

print("\n" + "=" * 55)
print("CHECK RESULT")
print("=" * 55)


if (
    test_id_duplicate == 0
    and missing_id == 0
):
    print("[PASS] ID unique và không missing.")
else:
    print("[FAIL] ID có duplicate hoặc missing.")


if (
    test_key_duplicate == 0
    and missing_date == 0
    and missing_store == 0
    and missing_item == 0
):
    print(
        "[PASS] "
        "(date, store_nbr, item_nbr) "
        "unique và không missing."
    )
else:
    print(
        "[FAIL] "
        "Composite key có duplicate hoặc missing."
    )


if (
    len(test_dates) == 16
    and test_min_date == "2017-08-16"
    and test_max_date == "2017-08-31"
):
    print(
        "[PASS] "
        "Test có đúng 16 ngày: "
        "2017-08-16 → 2017-08-31."
    )
else:
    print(
        "[WARNING] "
        "Date range không đúng kỳ vọng."
    )

print("\nSample test stores:")
print(sorted(test_stores)[:10])

print("\nSample test items:")
print(sorted(test_items)[:10])

print("\nTest dates:")
print(sorted(test_dates))


Chunk  1 checked | rows = 500,000
Chunk  2 checked | rows = 500,000
Chunk  3 checked | rows = 500,000
Chunk  4 checked | rows = 500,000
Chunk  5 checked | rows = 500,000
Chunk  6 checked | rows = 500,000
Chunk  7 checked | rows = 370,464

TEST INTEGRITY RESULT
Tổng số dòng                    : 3,370,464

--- DUPLICATE ---
Duplicate id                    : 0
Duplicate (date, store, item)   : 0

--- MISSING KEY VALUES ---
Missing id                      : 0
Missing date                    : 0
Missing store_nbr               : 0
Missing item_nbr                : 0

--- COVERAGE ---
Số store khác nhau              : 54
Số item khác nhau               : 3,901
Số ngày khác nhau               : 16

--- DATE RANGE ---
Ngày bắt đầu test               : 2017-08-16
Ngày kết thúc test              : 2017-08-31

CHECK RESULT
[PASS] ID unique và không missing.
[PASS] (date, store_nbr, item_nbr) unique và không missing.
[PASS] Test có đúng 16 ngày: 2017-08-16 → 2017-08-31.

Sample test stores:
[np.in

In [9]:
# CELL 09 — Train/Test/Reference coverage
# Mục tiêu:
# - Train item có tồn tại trong items.csv?
# - Test item có tồn tại trong items.csv?
# - Train store có tồn tại trong stores.csv?
# - Test store có tồn tại trong stores.csv?
# - Xác định item mới và pair (store,item) cold-start
#
# Lưu ý:
# Cell này sẽ đọc store_nbr từ toàn bộ train.csv theo chunks.
# Đây là một scan dữ liệu, nhưng không load toàn bộ train vào RAM.

# ---------------------------------
# 1. Các key từ reference tables
# ---------------------------------

reference_items = set(
    items["item_nbr"].dropna().astype(int)
)

reference_stores = set(
    stores["store_nbr"].dropna().astype(int)
)

# ---------------------------------
# 2. Train stores
# ---------------------------------

train_stores = set()

reader = (batch.to_pandas() for batch in pq.ParquetFile("../data/processed/train.parquet").iter_batches(batch_size=chunk_size))

for chunk in reader:
    train_stores.update(
        chunk["store_nbr"].unique()
    )

# ---------------------------------
# 3. Coverage
# ---------------------------------

train_item_missing_ref = train_items - reference_items
test_item_missing_ref = test_items - reference_items

train_store_missing_ref = train_stores - reference_stores
test_store_missing_ref = test_stores - reference_stores

# Item mới trong test
new_items_in_test = test_items - train_items

# Stores mới trong test
new_stores_in_test = test_stores - train_stores

print("=== REFERENCE COVERAGE ===")

print(
    f"Train items not in items.csv : "
    f"{len(train_item_missing_ref):,}"
)

print(
    f"Test items not in items.csv  : "
    f"{len(test_item_missing_ref):,}"
)

print(
    f"Train stores not in stores.csv: "
    f"{len(train_store_missing_ref):,}"
)

print(
    f"Test stores not in stores.csv : "
    f"{len(test_store_missing_ref):,}"
)

print("\n=== TRAIN → TEST COVERAGE ===")

print(
    f"Items only appearing in test  : "
    f"{len(new_items_in_test):,}"
)

print(
    f"Stores only appearing in test : "
    f"{len(new_stores_in_test):,}"
)

# ---------------------------------
# 4. Chi tiết nếu có vấn đề
# ---------------------------------

if train_item_missing_ref:
    print("\n[WARNING] Train item không tồn tại trong items.csv:")
    print(sorted(train_item_missing_ref)[:20])

if test_item_missing_ref:
    print("\n[WARNING] Test item không tồn tại trong items.csv:")
    print(sorted(test_item_missing_ref)[:20])

if train_store_missing_ref:
    print("\n[WARNING] Train store không tồn tại trong stores.csv:")
    print(sorted(train_store_missing_ref)[:20])

if test_store_missing_ref:
    print("\n[WARNING] Test store không tồn tại trong stores.csv:")
    print(sorted(test_store_missing_ref)[:20])

print("\n=== EXPECTED ===")

if not train_item_missing_ref and not test_item_missing_ref:
    print("[PASS] Tất cả item đều có reference trong items.csv.")
else:
    print("[FAIL] Có item không có reference.")

if not train_store_missing_ref and not test_store_missing_ref:
    print("[PASS] Tất cả store đều có reference trong stores.csv.")
else:
    print("[FAIL] Có store không có reference.")

print(
    "\nLưu ý: 'item chỉ xuất hiện trong test' khác với "
    "'store-item pair chưa từng xuất hiện trong train'."
)


=== REFERENCE COVERAGE ===
Train items not in items.csv : 0
Test items not in items.csv  : 0
Train stores not in stores.csv: 0
Test stores not in stores.csv : 0

=== TRAIN → TEST COVERAGE ===
Items only appearing in test  : 60
Stores only appearing in test : 0

=== EXPECTED ===
[PASS] Tất cả item đều có reference trong items.csv.
[PASS] Tất cả store đều có reference trong stores.csv.

Lưu ý: 'item chỉ xuất hiện trong test' khác với 'store-item pair chưa từng xuất hiện trong train'.


In [10]:
# CELL 10 — Test grid completeness
# Mục tiêu:
# - Xác nhận test có đúng số ngày dự báo cho mỗi store-item
# - Kiểm tra từng composite key (date, store, item) unique
# - Kiểm tra mỗi store-item pair có đúng 16 ngày
# - Kiểm tra date range liên tục

TEST_EXPECTED_START = "2017-08-16"
TEST_EXPECTED_END = "2017-08-31"
EXPECTED_HORIZON = 16

pair_counts = {}
date_counts = {}

test_reader = pd.read_csv(
    os.path.join(RAW_DIR, "test.csv"),
    usecols=[
        "date",
        "store_nbr",
        "item_nbr"
    ],
    chunksize=500_000,
    low_memory=False
)

for chunk in test_reader:

    # Pair = store + item
    pair_series = (
        chunk.groupby(
            ["store_nbr", "item_nbr"]
        ).size()
    )

    for pair, count in pair_series.items():
        pair_counts[pair] = (
            pair_counts.get(pair, 0) + int(count)
        )

    # Date count
    day_series = chunk.groupby("date").size()

    for date, count in day_series.items():
        date_counts[date] = (
            date_counts.get(date, 0) + int(count)
        )

pair_count_series = pd.Series(pair_counts)
date_count_series = pd.Series(date_counts)

expected_pairs = len(test_stores) * len(test_items)

print("=== TEST GRID ===")
print(f"Stores trong test             : {len(test_stores):,}")
print(f"Items trong test              : {len(test_items):,}")
print(f"Store-item pairs theo tập hợp : {expected_pairs:,}")
print(f"Store-item pairs thực tế      : {len(pair_count_series):,}")
print(f"Horizon kỳ vọng mỗi pair      : {EXPECTED_HORIZON} ngày")

# Kiểm tra số dòng/pair
bad_pairs = pair_count_series[
    pair_count_series != EXPECTED_HORIZON
]

print(
    f"\nPair không có đúng "
    f"{EXPECTED_HORIZON} ngày : {len(bad_pairs):,}"
)

if len(bad_pairs):
    print("\nMột số pair bất thường:")
    display(
        bad_pairs
        .sort_values()
        .head(20)
        .rename("n_rows")
        .reset_index()
    )

# Kiểm tra ngày
expected_test_dates = pd.date_range(
    TEST_EXPECTED_START,
    TEST_EXPECTED_END,
    freq="D"
)

actual_test_dates = pd.to_datetime(
    date_count_series.index
)

missing_test_dates = expected_test_dates.difference(
    actual_test_dates
)

unexpected_test_dates = actual_test_dates[
    ~actual_test_dates.isin(expected_test_dates)
]

print(
    f"\nNgày kỳ vọng                  : "
    f"{len(expected_test_dates)}"
)

print(
    f"Ngày thực tế                  : "
    f"{len(actual_test_dates)}"
)

print(
    f"Ngày kỳ vọng nhưng bị thiếu   : "
    f"{len(missing_test_dates)}"
)

print(
    f"Ngày nằm ngoài khoảng kỳ vọng : "
    f"{len(unexpected_test_dates)}"
)

# Kiểm tra mỗi ngày có cùng số lượng rows
print("\n=== ROWS PER DATE ===")
display(
    date_count_series
    .sort_index()
    .rename("n_rows")
    .to_frame()
)

# Tổng kết
grid_ok = (
    len(bad_pairs) == 0
    and len(missing_test_dates) == 0
    and len(unexpected_test_dates) == 0
)

print("\n=== FINAL RESULT OF CELL 10 ===")

if grid_ok:
    print("[PASS] Test grid có đủ 16 ngày cho mọi store-item pair.")
    print("[PASS] Khoảng thời gian test đúng 2017-08-16 → 2017-08-31.")
else:
    print("[FAIL] Test grid chưa đạt cấu trúc kỳ vọng.")
    print("Cần xem các bảng bất thường ở phía trên.")


=== TEST GRID ===
Stores trong test             : 54
Items trong test              : 3,901
Store-item pairs theo tập hợp : 210,654
Store-item pairs thực tế      : 210,654
Horizon kỳ vọng mỗi pair      : 16 ngày

Pair không có đúng 16 ngày : 0

Ngày kỳ vọng                  : 16
Ngày thực tế                  : 16
Ngày kỳ vọng nhưng bị thiếu   : 0
Ngày nằm ngoài khoảng kỳ vọng : 0

=== ROWS PER DATE ===


,n_rows
2017-08-16,210654
2017-08-17,210654
2017-08-18,210654
2017-08-19,210654
2017-08-20,210654
2017-08-21,210654
2017-08-22,210654
2017-08-23,210654
2017-08-24,210654
2017-08-25,210654



=== FINAL RESULT OF CELL 10 ===
[PASS] Test grid có đủ 16 ngày cho mọi store-item pair.
[PASS] Khoảng thời gian test đúng 2017-08-16 → 2017-08-31.


In [11]:
# CELL 11 — Investigate missing train dates with holidays
# Mục tiêu:
# - Đối chiếu các ngày bị thiếu trong train với holidays_events.csv
# - Kiểm tra loại holiday/event, locale, locale_name, transferred
# - Kiểm tra các ngày liền trước/sau để xem có pattern bất thường
#
# Không sửa dữ liệu train ở cell này.
# Không kết luận "store đóng cửa" chỉ từ việc thiếu row;
# ta chỉ ghi nhận evidence về holiday/closure pattern.

# =========================================================
# 1. Xác định các ngày bị thiếu từ daily
# =========================================================

expected_dates = pd.date_range(
    daily.index.min(),
    daily.index.max(),
    freq="D"
)

missing_dates = expected_dates.difference(daily.index)

print("=== MISSING TRAIN DATES ===")
print(f"Số ngày bị thiếu: {len(missing_dates):,}")

if len(missing_dates) == 0:
    print("[PASS] Không có ngày thiếu trong train.")
else:
    display(
        pd.DataFrame({
            "missing_date": missing_dates
        })
    )


# =========================================================
# 2. Đối chiếu với holidays_events.csv
# =========================================================

holidays_check = holidays.copy()

holidays_check["date"] = pd.to_datetime(
    holidays_check["date"],
    errors="coerce"
)

missing_holidays = holidays_check[
    holidays_check["date"].isin(missing_dates)
].copy()

missing_holidays = missing_holidays.sort_values(
    ["date", "locale", "locale_name", "type"]
)

print("\n=== HOLIDAYS / EVENTS ON MISSING DATES ===")

if missing_holidays.empty:
    print("[WARNING] Không tìm thấy holiday/event tương ứng.")
else:
    display(
        missing_holidays[
            [
                "date",
                "type",
                "locale",
                "locale_name",
                "description",
                "transferred"
            ]
        ].reset_index(drop=True)
    )


# =========================================================
# 3. Tổng hợp từng ngày thiếu
# =========================================================

print("\n=== SUMMARY BY MISSING DATE ===")

if not missing_holidays.empty:

    missing_summary = (
        missing_holidays
        .groupby("date")
        .agg(
            n_events=("description", "size"),
            event_types=("type", lambda x: ", ".join(sorted(x.astype(str).unique()))),
            locales=("locale", lambda x: ", ".join(sorted(x.astype(str).unique()))),
            locale_names=(
                "locale_name",
                lambda x: ", ".join(sorted(x.astype(str).unique()))
            ),
            descriptions=(
                "description",
                lambda x: " | ".join(sorted(x.astype(str).unique()))
            ),
            any_transferred=("transferred", "any"),
        )
    )

    display(missing_summary.reset_index())


# =========================================================
# 4. Kiểm tra pattern trước/sau ngày thiếu
# =========================================================
# Dùng daily profile đã tạo ở Cell 06.
#
# Ta xem:
# - số dòng
# - số store
# - tổng sales
#
# của ngày trước và sau mỗi ngày thiếu.

print("\n=== BEFORE / AFTER PATTERN ===")

rows = []

for d in missing_dates:

    prev_day = d - pd.Timedelta(days=1)
    next_day = d + pd.Timedelta(days=1)

    prev_info = daily.loc[prev_day] if prev_day in daily.index else None
    next_info = daily.loc[next_day] if next_day in daily.index else None

    rows.append({
        "missing_date": d,
        "missing_day": d.day_name(),

        "prev_date": prev_day,
        "prev_n_rows": (
            prev_info["n_rows"]
            if prev_info is not None
            else np.nan
        ),
        "prev_n_stores": (
            prev_info["n_stores"]
            if prev_info is not None
            else np.nan
        ),
        "prev_sales_sum": (
            prev_info["sales_sum"]
            if prev_info is not None
            else np.nan
        ),

        "next_date": next_day,
        "next_n_rows": (
            next_info["n_rows"]
            if next_info is not None
            else np.nan
        ),
        "next_n_stores": (
            next_info["n_stores"]
            if next_info is not None
            else np.nan
        ),
        "next_sales_sum": (
            next_info["sales_sum"]
            if next_info is not None
            else np.nan
        ),
    })

before_after = pd.DataFrame(rows)

display(before_after)


# =========================================================
# 5. Kiểm tra các ngày thiếu có phải cùng một pattern?
# =========================================================

print("\n=== INTERPRETATION ===")

if len(missing_dates) == 0:
    print("[PASS] Không có missing date cần điều tra.")

else:

    if not missing_holidays.empty:
        print(
            "[INFO] Một hoặc nhiều missing dates có holiday/event "
            "trong holidays_events.csv."
        )
    else:
        print(
            "[WARNING] Missing dates không có holiday/event "
            "tương ứng trong holidays_events.csv."
        )

    # Kiểm tra xem tất cả missing date có cùng ngày/tháng hay không
    month_day_pairs = sorted(
        set(
            (d.month, d.day)
            for d in missing_dates
        )
    )

    print(
        f"\nCác month-day của missing dates: "
        f"{month_day_pairs}"
    )

    if len(month_day_pairs) == 1:
        print(
            "[INFO] Các missing dates có cùng month/day "
            "qua nhiều năm → có pattern theo lịch."
        )
    else:
        print(
            "[INFO] Missing dates không có cùng month/day."
        )

    print(
        "\nLưu ý: train không có row ở một ngày chỉ chứng minh "
        "rằng không có quan sát sales trong train cho ngày đó. "
        "Cell này không tự động kết luận nguyên nhân là 'store đóng cửa'."
    )


=== MISSING TRAIN DATES ===
Số ngày bị thiếu: 4


,missing_date
0,2013-12-25
1,2014-12-25
2,2015-12-25
3,2016-12-25



=== HOLIDAYS / EVENTS ON MISSING DATES ===


,date,type,locale,locale_name,description,transferred
0,2013-12-25,Holiday,National,Ecuador,Navidad,False
1,2014-12-25,Holiday,National,Ecuador,Navidad,False
2,2015-12-25,Holiday,National,Ecuador,Navidad,False
3,2016-12-25,Holiday,National,Ecuador,Navidad,False



=== SUMMARY BY MISSING DATE ===


,date,n_events,event_types,locales,locale_names,descriptions,any_transferred
0,2013-12-25,1,Holiday,National,Ecuador,Navidad,False
1,2014-12-25,1,Holiday,National,Ecuador,Navidad,False
2,2015-12-25,1,Holiday,National,Ecuador,Navidad,False
3,2016-12-25,1,Holiday,National,Ecuador,Navidad,False



=== BEFORE / AFTER PATTERN ===


,missing_date,missing_day,prev_date,prev_n_rows,prev_n_stores,prev_sales_sum,next_date,next_n_rows,next_n_stores,next_sales_sum
0,2013-12-25,Wednesday,2013-12-24,52078.0,47.0,6.206633e+05,2013-12-26,50890.0,47.0,4.707312e+05
1,2014-12-25,Thursday,2014-12-24,75505.0,48.0,9.659829e+05,2014-12-26,75879.0,48.0,8.720984e+05
2,2015-12-25,Friday,2015-12-24,97660.0,53.0,1.083680e+06,2015-12-26,98456.0,53.0,9.467252e+05
3,2016-12-25,Sunday,2016-12-24,111289.0,53.0,1.246838e+06,2016-12-26,108352.0,53.0,1.032771e+06



=== INTERPRETATION ===
[INFO] Một hoặc nhiều missing dates có holiday/event trong holidays_events.csv.

Các month-day của missing dates: [(12, 25)]
[INFO] Các missing dates có cùng month/day qua nhiều năm → có pattern theo lịch.

Lưu ý: train không có row ở một ngày chỉ chứng minh rằng không có quan sát sales trong train cho ngày đó. Cell này không tự động kết luận nguyên nhân là 'store đóng cửa'.


In [12]:
# CELL 12 — Oil integrity (revised, variable-safe)
# Mục tiêu:
# - Kiểm tra date của oil.csv
# - Kiểm tra duplicate date
# - Phân biệt:
#     + missing date
#     + date có nhưng dcoilwtico = NaN
# - Kiểm tra coverage so với train + test
#
# Lưu ý:
# - Không dùng tên "missing_dates" vì biến này đã được dùng
#   cho missing dates của train.
# - Không fill/impute oil ở bước Checking.

oil_check = oil.copy()

oil_check["date"] = pd.to_datetime(
    oil_check["date"],
    errors="coerce"
)

# =========================================================
# 1. BASIC INTEGRITY
# =========================================================

oil_duplicate_date_count = int(
    oil_check["date"].duplicated().sum()
)

oil_missing_date_count = int(
    oil_check["date"].isna().sum()
)

oil_missing_value_count = int(
    oil_check["dcoilwtico"].isna().sum()
)

oil_negative_value_count = int(
    (oil_check["dcoilwtico"] < 0).sum()
)

oil_min_date = oil_check["date"].min()
oil_max_date = oil_check["date"].max()

print("=== BASIC OIL INTEGRITY ===")

print(
    f"Số dòng                    : "
    f"{len(oil_check):,}"
)

print(
    f"Duplicate date             : "
    f"{oil_duplicate_date_count:,}"
)

print(
    f"Missing date               : "
    f"{oil_missing_date_count:,}"
)

print(
    f"Missing dcoilwtico         : "
    f"{oil_missing_value_count:,}"
)

print(
    f"Negative dcoilwtico        : "
    f"{oil_negative_value_count:,}"
)

print(
    f"Ngày đầu                   : "
    f"{oil_min_date.date()}"
)

print(
    f"Ngày cuối                  : "
    f"{oil_max_date.date()}"
)


# =========================================================
# 2. CÁC DÒNG NaN CỦA OIL
# =========================================================

oil_missing_rows = oil_check[
    oil_check["dcoilwtico"].isna()
].copy()

print("\n=== MISSING OIL VALUES ===")

print(
    f"Số ngày có date nhưng thiếu giá dầu: "
    f"{len(oil_missing_rows):,}"
)

display(
    oil_missing_rows.head(20)
)


# =========================================================
# 3. WEEKDAY / WEEKEND STRUCTURE
# =========================================================

oil_check["day_of_week"] = (
    oil_check["date"].dt.day_name()
)

oil_check["weekday_num"] = (
    oil_check["date"].dt.dayofweek
)

weekday_counts = (
    oil_check["weekday_num"]
    .value_counts()
    .sort_index()
)

weekday_table = pd.DataFrame({
    "day_of_week": [
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ],
    "n_rows": [
        weekday_counts.get(i, 0)
        for i in range(7)
    ]
})

print("\n=== OIL DATE DISTRIBUTION BY WEEKDAY ===")

display(weekday_table)


# =========================================================
# 4. PROJECT DATE COVERAGE
# =========================================================

project_start = pd.Timestamp("2013-01-01")
project_end = pd.Timestamp("2017-08-31")

oil_actual_dates = set(
    oil_check["date"].dropna()
)

project_dates = pd.date_range(
    project_start,
    project_end,
    freq="D"
)

oil_dates_without_row = [
    d
    for d in project_dates
    if d not in oil_actual_dates
]

print("\n=== PROJECT DATE COVERAGE ===")

print(
    f"Project start              : "
    f"{project_start.date()}"
)

print(
    f"Project end                : "
    f"{project_end.date()}"
)

print(
    f"Số ngày calendar project   : "
    f"{len(project_dates):,}"
)

print(
    f"Số ngày xuất hiện trong oil: "
    f"{len(oil_actual_dates):,}"
)

print(
    f"Ngày không có row trong oil: "
    f"{len(oil_dates_without_row):,}"
)


# =========================================================
# 5. TRAIN + TEST COVERAGE
# =========================================================

train_start = daily.index.min()
train_end = daily.index.max()

test_start = pd.Timestamp("2017-08-16")
test_end = pd.Timestamp("2017-08-31")

oil_covers_train = (
    oil_min_date <= train_start
    and oil_max_date >= train_end
)

oil_covers_test = (
    oil_min_date <= test_start
    and oil_max_date >= test_end
)

print("\n=== TRAIN / TEST COVERAGE ===")

print(
    f"Oil phủ train timeframe : "
    f"{oil_covers_train}"
)

print(
    f"Oil phủ test timeframe  : "
    f"{oil_covers_test}"
)


# =========================================================
# 6. FINAL CHECK
# =========================================================

oil_ok = (
    oil_duplicate_date_count == 0
    and oil_missing_date_count == 0
    and oil_negative_value_count == 0
    and oil_covers_train
    and oil_covers_test
)

print("\n=== CHECK RESULT ===")

if oil_ok:
    print(
        "[PASS] oil.csv hợp lệ về cấu trúc và coverage."
    )
else:
    print(
        "[WARNING] oil.csv còn điểm cần xem xét."
    )

print("\n[GHI CHÚ]")

print(
    f"- Có {oil_missing_value_count:,} ngày có date "
    "nhưng dcoilwtico bị NaN."
)

print(
    "- Không tự động coi NaN là lỗi."
)

print(
    "- Không tự động fill NaN ở bước Checking."
)

print(
    "- Không tự động coi Oil là feature bắt buộc."
)


=== BASIC OIL INTEGRITY ===
Số dòng                    : 1,218
Duplicate date             : 0
Missing date               : 0
Missing dcoilwtico         : 43
Negative dcoilwtico        : 0
Ngày đầu                   : 2013-01-01
Ngày cuối                  : 2017-08-31

=== MISSING OIL VALUES ===
Số ngày có date nhưng thiếu giá dầu: 43


,date,dcoilwtico
0,2013-01-01,NaN
14,2013-01-21,NaN
34,2013-02-18,NaN
63,2013-03-29,NaN
104,2013-05-27,NaN
132,2013-07-04,NaN
174,2013-09-02,NaN
237,2013-11-28,NaN
256,2013-12-25,NaN
261,2014-01-01,NaN



=== OIL DATE DISTRIBUTION BY WEEKDAY ===


,day_of_week,n_rows
0,Monday,243
1,Tuesday,244
2,Wednesday,244
3,Thursday,244
4,Friday,243
5,Saturday,0
6,Sunday,0



=== PROJECT DATE COVERAGE ===
Project start              : 2013-01-01
Project end                : 2017-08-31
Số ngày calendar project   : 1,704
Số ngày xuất hiện trong oil: 1,218
Ngày không có row trong oil: 486

=== TRAIN / TEST COVERAGE ===
Oil phủ train timeframe : True
Oil phủ test timeframe  : True

=== CHECK RESULT ===
[PASS] oil.csv hợp lệ về cấu trúc và coverage.

[GHI CHÚ]
- Có 43 ngày có date nhưng dcoilwtico bị NaN.
- Không tự động coi NaN là lỗi.
- Không tự động fill NaN ở bước Checking.
- Không tự động coi Oil là feature bắt buộc.


In [13]:
# CELL 13 — Transactions integrity
# Mục tiêu:
# - Kiểm tra key (date, store_nbr)
# - Kiểm tra missing
# - Kiểm tra giá trị transactions
# - Kiểm tra store có tồn tại trong stores.csv
# - Xác định phạm vi thời gian của transactions
#
# Lưu ý:
# transactions là dữ liệu lịch sử.
# Không được giả định rằng transactions có sẵn cho các ngày test tương lai.

transactions_check = transactions.copy()

transactions_check["date"] = pd.to_datetime(
    transactions_check["date"],
    errors="coerce"
)

# ---------------------------------------------------------
# 1. Basic integrity
# ---------------------------------------------------------

transactions_duplicate = int(
    transactions_check.duplicated(
        ["date", "store_nbr"]
    ).sum()
)

transactions_missing_date = int(
    transactions_check["date"].isna().sum()
)

transactions_missing_store = int(
    transactions_check["store_nbr"].isna().sum()
)

transactions_missing_value = int(
    transactions_check["transactions"].isna().sum()
)

transactions_negative = int(
    (transactions_check["transactions"] < 0).sum()
)

# ---------------------------------------------------------
# 2. Store reference
# ---------------------------------------------------------

transaction_stores = set(
    transactions_check["store_nbr"]
    .dropna()
    .astype(int)
)

reference_stores = set(
    stores["store_nbr"]
    .dropna()
    .astype(int)
)

transaction_store_not_reference = (
    transaction_stores - reference_stores
)

# ---------------------------------------------------------
# 3. Date range
# ---------------------------------------------------------

transactions_min_date = (
    transactions_check["date"].min()
)

transactions_max_date = (
    transactions_check["date"].max()
)

# ---------------------------------------------------------
# 4. Store coverage by date
# ---------------------------------------------------------

store_day_counts = (
    transactions_check
    .groupby("date")["store_nbr"]
    .nunique()
)

print("=== TRANSACTIONS INTEGRITY ===")

print(
    f"Số dòng                       : "
    f"{len(transactions_check):,}"
)

print(
    f"Duplicate (date, store)       : "
    f"{transactions_duplicate:,}"
)

print(
    f"Missing date                  : "
    f"{transactions_missing_date:,}"
)

print(
    f"Missing store_nbr             : "
    f"{transactions_missing_store:,}"
)

print(
    f"Missing transactions          : "
    f"{transactions_missing_value:,}"
)

print(
    f"Negative transactions         : "
    f"{transactions_negative:,}"
)

print(
    f"Số store khác nhau            : "
    f"{len(transaction_stores):,}"
)

print(
    f"Store không có trong stores   : "
    f"{len(transaction_store_not_reference):,}"
)

print(
    f"Ngày bắt đầu                  : "
    f"{transactions_min_date.date()}"
)

print(
    f"Ngày kết thúc                 : "
    f"{transactions_max_date.date()}"
)

print("\n=== STORE COVERAGE BY DATE ===")

print(
    f"Store/date thấp nhất          : "
    f"{store_day_counts.min()}"
)

print(
    f"Store/date cao nhất           : "
    f"{store_day_counts.max()}"
)

display(
    store_day_counts
    .describe()
    .to_frame("value")
)

if transaction_store_not_reference:
    print(
        "\n[WARNING] Có store trong transactions "
        "không tồn tại trong stores.csv:"
    )
    print(sorted(transaction_store_not_reference))

transactions_ok = (
    transactions_duplicate == 0
    and transactions_missing_date == 0
    and transactions_missing_store == 0
    and transactions_missing_value == 0
    and transactions_negative == 0
    and len(transaction_store_not_reference) == 0
)

print("\n=== CHECK RESULT ===")

if transactions_ok:
    print(
        "[PASS] transactions.csv có key hợp lệ, "
        "không missing và store đều có reference."
    )
else:
    print(
        "[WARNING] transactions.csv còn điểm cần xem xét."
    )

print(
    "\nLưu ý: transactions_max_date có thể kết thúc tại "
    "2017-08-15 và không phải lỗi; không có transaction "
    "thực tế cho test tương lai."
)


=== TRANSACTIONS INTEGRITY ===
Số dòng                       : 83,488
Duplicate (date, store)       : 0
Missing date                  : 0
Missing store_nbr             : 0
Missing transactions          : 0
Negative transactions         : 0
Số store khác nhau            : 54
Store không có trong stores   : 0
Ngày bắt đầu                  : 2013-01-01
Ngày kết thúc                 : 2017-08-15

=== STORE COVERAGE BY DATE ===
Store/date thấp nhất          : 1
Store/date cao nhất           : 54


,value
count,1682.000000
mean,49.636147
std,3.832710
min,1.000000
25%,47.000000
50%,49.000000
75%,53.000000
max,54.000000



=== CHECK RESULT ===
[PASS] transactions.csv có key hợp lệ, không missing và store đều có reference.

Lưu ý: transactions_max_date có thể kết thúc tại 2017-08-15 và không phải lỗi; không có transaction thực tế cho test tương lai.


In [14]:
# CELL 14 — Sample submission integrity
# Mục tiêu:
# - Kiểm tra sample_submission có cùng số dòng với test
# - Kiểm tra ID của sample_submission khớp TEST theo đúng thứ tự
# - Kiểm tra unit_sales không missing
#
# Đọc theo chunks để không phải giữ đồng thời toàn bộ
# sample_submission + test trong RAM.

sample_rows = 0
test_rows_for_sample = 0

sample_missing_prediction = 0
sample_order_mismatch = 0

sample_ok = True

test_reader = pd.read_csv(
    os.path.join(RAW_DIR, "test.csv"),
    usecols=["id"],
    dtype={"id": "int64"},
    chunksize=500_000,
    low_memory=False
)

sample_reader = pd.read_csv(
    os.path.join(RAW_DIR, "sample_submission.csv"),
    usecols=["id", "unit_sales"],
    dtype={
        "id": "int64",
        "unit_sales": "float64"
    },
    chunksize=500_000,
    low_memory=False
)

from itertools import zip_longest

for pair in zip_longest(
    test_reader,
    sample_reader,
    fillvalue=None
):

    test_chunk, sample_chunk = pair

    # Test hết trước sample
    if test_chunk is None or sample_chunk is None:
        sample_order_mismatch += 1
        sample_ok = False
        continue

    test_rows_for_sample += len(test_chunk)
    sample_rows += len(sample_chunk)

    # Số dòng trong hai chunk phải giống nhau
    if len(test_chunk) != len(sample_chunk):
        sample_order_mismatch += 1
        sample_ok = False
        continue

    # ID phải giống hoàn toàn, đúng thứ tự
    if not np.array_equal(
        test_chunk["id"].to_numpy(),
        sample_chunk["id"].to_numpy()
    ):
        sample_order_mismatch += 1
        sample_ok = False

    # Prediction không được missing
    sample_missing_prediction += int(
        sample_chunk["unit_sales"].isna().sum()
    )


print("=== SAMPLE SUBMISSION INTEGRITY ===")

print(
    f"Số dòng test                  : "
    f"{test_rows_for_sample:,}"
)

print(
    f"Số dòng sample_submission     : "
    f"{sample_rows:,}"
)

print(
    f"Chunk/order mismatch          : "
    f"{sample_order_mismatch:,}"
)

print(
    f"Missing unit_sales            : "
    f"{sample_missing_prediction:,}"
)

if (
    sample_rows == test_rows_for_sample
    and sample_order_mismatch == 0
    and sample_missing_prediction == 0
):
    sample_ok = True

    print(
        "\n[PASS] sample_submission có cùng số dòng "
        "và ID khớp TEST theo đúng thứ tự."
    )
else:
    sample_ok = False

    print(
        "\n[FAIL] sample_submission không khớp TEST."
    )


=== SAMPLE SUBMISSION INTEGRITY ===
Số dòng test                  : 3,370,464
Số dòng sample_submission     : 3,370,464
Chunk/order mismatch          : 0
Missing unit_sales            : 0

[PASS] sample_submission có cùng số dòng và ID khớp TEST theo đúng thứ tự.


In [15]:
# CELL 15 — Series history & cold-start analysis
# Mục tiêu:
# - Đếm số ngày quan sát của từng (store, item) trong train
# - Xác định test store-item pair có lịch sử hay không
# - Phân biệt:
#     1) item mới
#     2) store-item pair mới
# - Phân tích độ sparse của các series
#
# Không load toàn bộ train vào RAM.
#
# Vì train key đã được kiểm tra unique ở Cell 05,
# số row của một pair cũng chính là số ngày pair đó xuất hiện.

# Test hiện tại đã được chứng minh là:
# 54 stores × 3901 items × 16 ngày.
# Vì vậy tập pair test chính là Cartesian product.
test_pair_codes = {
    (int(store), int(item))
    for store in test_stores
    for item in test_items
}

pair_counts = {}

reader = (batch.to_pandas() for batch in pq.ParquetFile("../data/processed/train.parquet").iter_batches(batch_size=1_000_000))

for chunk in reader:

    pairs = list(
        zip(
            chunk["store_nbr"].astype(int),
            chunk["item_nbr"].astype(int)
        )
    )

    counts = pd.Series(pairs).value_counts()

    for pair, count in counts.items():
        pair_counts[pair] = (
            pair_counts.get(pair, 0)
            + int(count)
        )

train_pair_codes = set(pair_counts)

cold_start_pairs = (
    test_pair_codes - train_pair_codes
)

known_test_pairs = (
    test_pair_codes & train_pair_codes
)

history_lengths = pd.Series(
    list(pair_counts.values()),
    name="observed_days"
)

print("\n=== SERIES HISTORY ===")

print(
    f"Số series (store,item) trong train : "
    f"{len(train_pair_codes):,}"
)

print(
    f"Số test store-item pairs           : "
    f"{len(test_pair_codes):,}"
)

print(
    f"Test pairs có history              : "
    f"{len(known_test_pairs):,}"
)

print(
    f"Test pairs chưa từng xuất hiện     : "
    f"{len(cold_start_pairs):,}"
)

print(
    f"Cold-start pair rate               : "
    f"{len(cold_start_pairs) / len(test_pair_codes):.2%}"
)

print("\n=== HISTORY LENGTH ===")

display(
    history_lengths.describe(
        percentiles=[
            0.01,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.99
        ]
    ).to_frame()
)

print("\n=== TEST-ONLY ITEMS ===")

test_only_items = test_items - train_items

print(
    f"Item chỉ xuất hiện trong test : "
    f"{len(test_only_items):,}"
)

if test_only_items:
    print(
        "Một số item test-only:",
        sorted(test_only_items)[:20]
    )

print("\n=== INTERPRETATION ===")
print(
    "- Cold-start pair khác hoàn toàn với item mới."
)
print(
    "- Một item có thể đã tồn tại trong train nhưng "
    "một store-item pair cụ thể lại chưa từng xuất hiện."
)
print(
    "- Phân bố observed_days cho biết độ sparse của các series "
    "và sẽ quyết định cách tạo lag/rolling sau này."
)



=== SERIES HISTORY ===
Số series (store,item) trong train : 174,685
Số test store-item pairs           : 210,654
Test pairs có history              : 166,213
Test pairs chưa từng xuất hiện     : 44,441
Cold-start pair rate               : 21.10%

=== HISTORY LENGTH ===


,observed_days
count,174685.000000
mean,718.419097
std,514.831287
min,1.000000
1%,3.000000
10%,79.000000
25%,245.000000
50%,651.000000
75%,1133.000000
90%,1510.000000



=== TEST-ONLY ITEMS ===
Item chỉ xuất hiện trong test : 60
Một số item test-only: [np.int32(313094), np.int32(1464607), np.int32(1467088), np.int32(1467093), np.int32(2015646), np.int32(2109969), np.int32(2112280), np.int32(2112303), np.int32(2112461), np.int32(2112462), np.int32(2112607), np.int32(2117783), np.int32(2118397), np.int32(2120180), np.int32(2120216), np.int32(2120506), np.int32(2120681), np.int32(2120768), np.int32(2121319), np.int32(2124705)]

=== INTERPRETATION ===
- Cold-start pair khác hoàn toàn với item mới.
- Một item có thể đã tồn tại trong train nhưng một store-item pair cụ thể lại chưa từng xuất hiện.
- Phân bố observed_days cho biết độ sparse của các series và sẽ quyết định cách tạo lag/rolling sau này.


In [16]:
# CELL 16 — Validation window & coverage
# Mục tiêu:
# - Xác nhận validation = 16 ngày liên tiếp trước test
# - Kiểm tra weekday distribution
# - Đo coverage của train trên test store-item universe
# - Kiểm tra số pair quan sát mỗi ngày
# - Kiểm tra promotion missing/true trong validation
#
# Quan trọng:
# Không fill missing unit_sales thành 0 ở bước này.
# Chỉ đo coverage của dữ liệu thực tế.

VAL_START = pd.Timestamp("2017-07-31")
VAL_END = pd.Timestamp("2017-08-15")

TEST_START = pd.Timestamp("2017-08-16")
TEST_END = pd.Timestamp("2017-08-31")

validation_dates = pd.date_range(
    VAL_START,
    VAL_END,
    freq="D"
)

test_dates_exact = pd.date_range(
    TEST_START,
    TEST_END,
    freq="D"
)

print("=== VALIDATION / TEST WINDOWS ===")

print(
    f"Validation: {VAL_START.date()} → "
    f"{VAL_END.date()} "
    f"({len(validation_dates)} ngày)"
)

print(
    f"Test      : {TEST_START.date()} → "
    f"{TEST_END.date()} "
    f"({len(test_dates_exact)} ngày)"
)

# ---------------------------------------------------------
# Weekday distribution
# ---------------------------------------------------------

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

val_dow = (
    validation_dates
    .day_name()
    .value_counts()
    .reindex(weekday_order, fill_value=0)
)

test_dow = (
    test_dates_exact
    .day_name()
    .value_counts()
    .reindex(weekday_order, fill_value=0)
)

print("\n=== WEEKDAY DISTRIBUTION ===")

weekday_compare = pd.DataFrame({
    "validation": val_dow,
    "test": test_dow
})

display(weekday_compare)

# ---------------------------------------------------------
# Scan train validation window
# ---------------------------------------------------------

validation_daily_rows = {}

reader = (batch.to_pandas() for batch in pq.ParquetFile("../data/processed/train.parquet").iter_batches(batch_size=1_000_000))

for chunk in reader:

    chunk["date"] = pd.to_datetime(
        chunk["date"],
        errors="coerce"
    )

    mask = (
        (chunk["date"] >= VAL_START)
        & (chunk["date"] <= VAL_END)
    )

    vchunk = chunk.loc[mask].copy()

    if vchunk.empty:
        continue

    # Chỉ giữ universe store-item giống test
    vchunk = vchunk[
        vchunk["store_nbr"].isin(test_stores)
        & vchunk["item_nbr"].isin(test_items)
    ]

    if vchunk.empty:
        continue

    grouped = (
        vchunk
        .groupby("date")
        .agg(
            n_rows=("item_nbr", "size"),
            n_stores=("store_nbr", "unique"),
            n_items=("item_nbr", "unique"),
            sales_sum=("unit_sales", "sum"),
            n_negative=(
                "unit_sales",
                lambda x: int((x < 0).sum())
            ),
            n_promo_nan=(
                "onpromotion",
                lambda x: int(x.isna().sum())
            ),
            n_promo_true=(
                "onpromotion",
                lambda x: int((x == True).sum())
            ),
        )
    )

    for date, row in grouped.iterrows():

        if date not in validation_daily_rows:
            validation_daily_rows[date] = {
                "n_rows": 0,
                "stores_set": set(),
                "items_set": set(),
                "sales_sum": 0.0,
                "n_negative": 0,
                "n_promo_nan": 0,
                "n_promo_true": 0,
            }

        validation_daily_rows[date]["n_rows"] += int(row["n_rows"])
        validation_daily_rows[date]["stores_set"].update(row["n_stores"])
        validation_daily_rows[date]["items_set"].update(row["n_items"])
        validation_daily_rows[date]["sales_sum"] += float(
            row["sales_sum"]
        )
        validation_daily_rows[date]["n_negative"] += int(
            row["n_negative"]
        )
        validation_daily_rows[date]["n_promo_nan"] += int(
            row["n_promo_nan"]
        )
        validation_daily_rows[date]["n_promo_true"] += int(
            row["n_promo_true"]
        )

for k, v in validation_daily_rows.items():
    v["n_stores"] = len(v["stores_set"])
    v["n_items"] = len(v["items_set"])
    del v["stores_set"]
    del v["items_set"]

validation_daily = (
    pd.DataFrame.from_dict(
        validation_daily_rows,
        orient="index"
    )
    .sort_index()
)

validation_daily.index.name = "date"

EXPECTED_PAIR_COUNT = (
    len(test_stores) * len(test_items)
)

validation_daily["expected_pairs"] = (
    EXPECTED_PAIR_COUNT
)

validation_daily["missing_pairs"] = (
    validation_daily["expected_pairs"]
    - validation_daily["n_rows"]
)

validation_daily["coverage_rate"] = (
    validation_daily["n_rows"]
    / validation_daily["expected_pairs"]
)

# ---------------------------------------------------------
# Missing validation dates
# ---------------------------------------------------------

missing_validation_dates = (
    validation_dates
    .difference(validation_daily.index)
)

print("\n=== VALIDATION COVERAGE ===")

print(
    f"Expected store-item pairs/day : "
    f"{EXPECTED_PAIR_COUNT:,}"
)

print(
    f"Validation dates with data     : "
    f"{validation_daily.index.nunique():,}"
)

print(
    f"Validation dates missing       : "
    f"{len(missing_validation_dates):,}"
)

display(validation_daily)

validation_ok = (
    len(validation_dates) == 16
    and len(test_dates_exact) == 16
    and len(missing_validation_dates) == 0
    and validation_daily["n_rows"].gt(0).all()
)

print("\n=== CHECK RESULT ===")

if validation_ok:
    print(
        "[PASS] Validation có đủ 16 ngày và có dữ liệu "
        "trên test store-item universe."
    )
else:
    print(
        "[FAIL] Validation window chưa đạt cấu trúc kỳ vọng."
    )

print(
    "\nLưu ý: coverage_rate < 100% không phải lỗi. "
    "Nó phản ánh train là sparse dataset."
)


=== VALIDATION / TEST WINDOWS ===
Validation: 2017-07-31 → 2017-08-15 (16 ngày)
Test      : 2017-08-16 → 2017-08-31 (16 ngày)

=== WEEKDAY DISTRIBUTION ===


,validation,test
Monday,3,2
Tuesday,3,2
Wednesday,2,3
Thursday,2,3
Friday,2,2
Saturday,2,2
Sunday,2,2



=== VALIDATION COVERAGE ===
Expected store-item pairs/day : 210,654
Validation dates with data     : 16
Validation dates missing       : 0


,n_rows,sales_sum,n_negative,n_promo_nan,n_promo_true,n_stores,n_items,expected_pairs,missing_pairs,coverage_rate
date,,,,,,,,,,
2017-07-31,106235,8.851828e+05,7,0,8649,54,3741,210654,104419,0.504310
2017-08-01,109251,9.877818e+05,8,0,10150,54,3748,210654,101403,0.518628
2017-08-02,107280,9.638030e+05,9,0,16484,54,3735,210654,103374,0.509271
2017-08-03,102117,7.273975e+05,8,0,7859,54,3737,210654,108537,0.484762
2017-08-04,104060,8.271467e+05,5,0,14193,54,3754,210654,106594,0.493985
2017-08-05,110780,9.650906e+05,4,0,8684,54,3753,210654,99874,0.525886
2017-08-06,110750,1.049029e+06,8,0,9555,54,3755,210654,99904,0.525744
2017-08-07,103813,7.971979e+05,15,0,8209,54,3746,210654,106841,0.492813
2017-08-08,101114,7.176354e+05,18,0,10637,54,3747,210654,109540,0.480000



=== CHECK RESULT ===
[PASS] Validation có đủ 16 ngày và có dữ liệu trên test store-item universe.

Lưu ý: coverage_rate < 100% không phải lỗi. Nó phản ánh train là sparse dataset.


In [17]:
# CELL 17 — Temporal split / leakage readiness
# Mục tiêu:
# - Xác nhận train kết thúc ngay trước test
# - Validation nằm hoàn toàn trong train
# - Validation và test không overlap
# - Không có khoảng thời gian bị đảo ngược
#
# Cell này chưa tạo feature.
# Chỉ xác nhận cấu trúc thời gian để chuẩn bị modeling.

# ---------------------------------------------------------
# 1. Date boundaries
# ---------------------------------------------------------

train_max_date = pd.Timestamp(
    daily.index.max()
)

train_min_date = pd.Timestamp(
    daily.index.min()
)

print("=== TEMPORAL SPLIT ===")

print(
    f"Train start       : "
    f"{train_min_date.date()}"
)

print(
    f"Train end         : "
    f"{train_max_date.date()}"
)

print(
    f"Validation start  : "
    f"{VAL_START.date()}"
)

print(
    f"Validation end    : "
    f"{VAL_END.date()}"
)

print(
    f"Test start        : "
    f"{TEST_START.date()}"
)

print(
    f"Test end          : "
    f"{TEST_END.date()}"
)

# ---------------------------------------------------------
# 2. Relationship checks
# ---------------------------------------------------------

validation_inside_train = (
    VAL_START >= train_min_date
    and VAL_END <= train_max_date
)

test_after_train = (
    TEST_START > train_max_date
)

validation_before_test = (
    VAL_END < TEST_START
)

no_validation_test_overlap = (
    set(validation_dates)
    .isdisjoint(set(test_dates_exact))
)

expected_one_day_boundary = (
    TEST_START == train_max_date + pd.Timedelta(days=1)
)

print("\n=== RELATIONSHIPS ===")

print(
    f"Validation nằm trong train : "
    f"{validation_inside_train}"
)

print(
    f"Test sau train              : "
    f"{test_after_train}"
)

print(
    f"Validation trước test      : "
    f"{validation_before_test}"
)

print(
    f"Validation/Test không overlap: "
    f"{no_validation_test_overlap}"
)

print(
    f"Test bắt đầu đúng ngày sau train: "
    f"{expected_one_day_boundary}"
)

# ---------------------------------------------------------
# 3. Final split check
# ---------------------------------------------------------

temporal_split_ok = (
    validation_inside_train
    and test_after_train
    and validation_before_test
    and no_validation_test_overlap
    and expected_one_day_boundary
)

print("\n=== CHECK RESULT ===")

if temporal_split_ok:
    print(
        "[PASS] Temporal split hợp lệ cho time-series forecasting."
    )
else:
    print(
        "[FAIL] Temporal split có vấn đề."
    )

print("\n=== LEAKAGE RULE FOR NEXT STAGE ===")

print(
    "Feature dùng target history phải chỉ sử dụng dữ liệu "
    "trước thời điểm cần dự đoán."
)

print(
    "Ví dụ: dự đoán ngày t → lag_7 dùng sales của t-7, "
    "không dùng sales của t hoặc tương lai."
)

print(
    "Validation phải được xem là tương lai đối với training."
)


=== TEMPORAL SPLIT ===
Train start       : 2013-01-01
Train end         : 2017-08-15
Validation start  : 2017-07-31
Validation end    : 2017-08-15
Test start        : 2017-08-16
Test end          : 2017-08-31

=== RELATIONSHIPS ===
Validation nằm trong train : True
Test sau train              : True
Validation trước test      : True
Validation/Test không overlap: True
Test bắt đầu đúng ngày sau train: True

=== CHECK RESULT ===
[PASS] Temporal split hợp lệ cho time-series forecasting.

=== LEAKAGE RULE FOR NEXT STAGE ===
Feature dùng target history phải chỉ sử dụng dữ liệu trước thời điểm cần dự đoán.
Ví dụ: dự đoán ngày t → lag_7 dùng sales của t-7, không dùng sales của t hoặc tương lai.
Validation phải được xem là tương lai đối với training.


In [18]:
# CELL 18 — FINAL DATA CHECKING GATE
# Mục tiêu:
# - Tổng hợp tất cả integrity checks
# - Không đánh FAIL các đặc điểm dữ liệu đã được giải thích
# - Chỉ cho phép chuyển sang Feature Engineering khi
#   mọi integrity check bắt buộc đều PASS.

print("=" * 70)
print("FINAL DATA CHECKING")
print("=" * 70)


# =========================================================
# 1. RAW FILES
# =========================================================

files_ok = (
    len(missing) == 0
)


# =========================================================
# 2. REFERENCE TABLES
# =========================================================

reference_ok = (
    items_dup == 0
    and stores_dup == 0
    and oil_duplicate_date_count == 0
    and trans_dup == 0
)


# =========================================================
# 3. TRAIN
# =========================================================

train_ok = (
    sorted_ok
    and daily_profile.index.is_unique
    and dup_total == 0
)


# =========================================================
# 4. TEST
# =========================================================

test_ok = (
    test_id_duplicate == 0
    and test_key_duplicate == 0
    and missing_id == 0
    and missing_date == 0
    and missing_store == 0
    and missing_item == 0
)


# =========================================================
# 5. TRAIN / TEST / REFERENCE COVERAGE
# =========================================================

coverage_ok = (
    len(train_item_missing_ref) == 0
    and len(test_item_missing_ref) == 0
    and len(train_store_missing_ref) == 0
    and len(test_store_missing_ref) == 0
)


# =========================================================
# 6. TEST GRID
# =========================================================

test_grid_ok = (
    len(bad_pairs) == 0
    and len(missing_test_dates) == 0
    and len(unexpected_test_dates) == 0
)


# =========================================================
# 7. OIL
# =========================================================

oil_final_ok = oil_ok


# =========================================================
# 8. TRANSACTIONS
# =========================================================

transactions_final_ok = transactions_ok


# =========================================================
# 9. SAMPLE SUBMISSION
# =========================================================

sample_final_ok = sample_ok


# =========================================================
# 10. VALIDATION
# =========================================================

validation_final_ok = validation_ok


# =========================================================
# 11. TEMPORAL SPLIT
# =========================================================

temporal_final_ok = temporal_split_ok


# =========================================================
# SUMMARY
# =========================================================

checks = {
    "Raw files": files_ok,
    "Reference tables": reference_ok,
    "Train integrity": train_ok,
    "Test integrity": test_ok,
    "Train/Test reference coverage": coverage_ok,
    "Test grid": test_grid_ok,
    "Oil": oil_final_ok,
    "Transactions": transactions_final_ok,
    "Sample submission": sample_final_ok,
    "Validation window": validation_final_ok,
    "Temporal split": temporal_final_ok,
}


print("\n=== REQUIRED CHECKS ===")

for name, result in checks.items():

    print(
        f"[{'PASS' if result else 'FAIL'}] {name}"
    )


# =========================================================
# DOCUMENTED DATA CHARACTERISTICS
# =========================================================

print("\n" + "=" * 70)
print(
    "DOCUMENTED DATA CHARACTERISTICS — "
    "NOT AUTOMATIC ERRORS"
)
print("=" * 70)

# Train missing dates
print(
    f"- Missing train dates          : "
    f"{len(train_missing_dates):,}"
)

print(
    "- Các missing train dates đã được đối chiếu "
    "với holidays_events.csv."
)

print(
    "- Pattern đã xác minh: 25/12 là "
    "National Holiday 'Navidad'."
)

# Negative sales
print(
    f"- Negative unit_sales rows     : "
    f"{negative_sales:,}"
)

# Explicit zero
print(
    f"- Explicit unit_sales == 0     : "
    f"{zero_sales:,}"
)

# Promotion
print(
    f"- Missing onpromotion rows     : "
    f"{promo_missing:,}"
)

# Cold-start
print(
    f"- Test-only items              : "
    f"{len(test_only_items):,}"
)

print(
    f"- Test cold-start pairs        : "
    f"{len(cold_start_pairs):,}"
)

print(
    "- Các đặc điểm này không phải lỗi integrity; "
    "chúng sẽ được xử lý/đánh giá ở "
    "Feature Engineering hoặc Modeling."
)


# =========================================================
# FINAL DECISION
# =========================================================

all_required_pass = all(
    checks.values()
)

print("\n" + "=" * 70)
print("FINAL STATUS")
print("=" * 70)

if all_required_pass:

    print(
        "[PASS] DATA CHECKING HOÀN TẤT."
    )

    print(
        "\nCó thể chuyển sang:"
    )

    print(
        "EDA FINALIZATION → Forecasting Setup → "
        "Feature Engineering"
    )

else:

    print(
        "[FAIL] DATA CHECKING CHƯA HOÀN TẤT."
    )

    print(
        "\nCác mục FAIL:"
    )

    for name, result in checks.items():
        if not result:
            print(f" - {name}")

    print(
        "\nKhông nên chuyển sang Feature Engineering "
        "cho tới khi các mục FAIL được xử lý."
    )


FINAL DATA CHECKING

=== REQUIRED CHECKS ===
[PASS] Raw files
[PASS] Reference tables
[PASS] Train integrity
[PASS] Test integrity
[PASS] Train/Test reference coverage
[PASS] Test grid
[PASS] Oil
[PASS] Transactions
[PASS] Sample submission
[PASS] Validation window
[PASS] Temporal split

DOCUMENTED DATA CHARACTERISTICS — NOT AUTOMATIC ERRORS
- Missing train dates          : 4
- Các missing train dates đã được đối chiếu với holidays_events.csv.
- Pattern đã xác minh: 25/12 là National Holiday 'Navidad'.
- Negative unit_sales rows     : 7,795
- Explicit unit_sales == 0     : 0
- Missing onpromotion rows     : 21,657,651
- Test-only items              : 60
- Test cold-start pairs        : 44,441
- Các đặc điểm này không phải lỗi integrity; chúng sẽ được xử lý/đánh giá ở Feature Engineering hoặc Modeling.

FINAL STATUS
[PASS] DATA CHECKING HOÀN TẤT.

Có thể chuyển sang:
EDA FINALIZATION → Forecasting Setup → Feature Engineering
